# m02_l05_retries

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Finds the code companion. On Google Colab it downloads it once.
import glob, io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if not os.path.isdir("shopassist"):
    if os.path.isdir(os.path.join("..", "shopassist")):
        os.chdir("..")  # opened from the notebooks folder on your computer
    else:
        found = glob.glob("*-companion")  # kept: your edited answers stay
        if not found and "google.colab" in sys.modules:
            try:
                data = io.BytesIO(urllib.request.urlopen(URL).read())
            except OSError:  # network blocks the download: upload the zip
                from google.colab import files
                data = next(iter(files.upload()))
            zipfile.ZipFile(data).extractall(".")
            found = glob.glob("*-companion")
        if found:
            os.chdir(found[0])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Companion ready" if os.path.isdir("shopassist") else
      "Companion not found: see Lesson 0.1 of the book")


## Retry

In [ ]:
from shopassist.client import post_json, with_retries
from shopassist.mockserver import MockProvider

body = {"model": "mock", "max_tokens": 200, "system": "Be brief.",
        "messages": [{"role": "user", "content": "Order #1042?"}]}


def log(attempt, status, wait):
    print(f"attempt {attempt} failed with {status};",
          f"waiting {wait:.2f}s")


# The stand-in provider fails three times, then answers.
with MockProvider(fail_with=[429, 500, 503]) as server:
    def call():
        return post_json(server.url, body, "test-key")
    reply = with_retries(call, log=log, seed=7, sleep=lambda s: None)
    calls = len(server.requests)

print("succeeded after", calls, "calls:", reply["content"][0]["text"])
